In [ ]:
import pandas as pd
import numpy as np
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import os

In [ ]:
RANDOM_SEED = 42
LEARNING_RATE = 1e-3
LAMBDA = 1e-4
BATCH_SIZE = 64

In [ ]:
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
DATA_DIR = '../data/processed/'

X_train = np.load(DATA_DIR + 'X_train.npy')
X_val   = np.load(DATA_DIR + 'X_val.npy')
X_test  = np.load(DATA_DIR + 'X_test.npy')
y_train = np.load(DATA_DIR + 'y_train.npy')
y_val   = np.load(DATA_DIR + 'y_val.npy')
y_test  = np.load(DATA_DIR + 'y_test.npy')

X_train_t = torch.tensor(X_train, dtype = torch.float32)
X_val_t = torch.tensor(X_val, dtype = torch.float32)
X_test_t = torch.tensor(X_test, dtype = torch.float32)

y_train_t = torch.tensor(y_train, dtype = torch.float32).unsqueeze(1)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

In [ ]:
class Housingdata(Dataset):
    def __init__(self, X,y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
training_dataset = Housingdata(X_train_t, y_train_t)

In [ ]:
val_dataset = Housingdata(X_val_t, y_val_t)
test_dataset = Housingdata(X_test_t, y_test_t)

In [ ]:
train_loader = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle= False, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self, n_features):
        super(LinearRegression,self).__init__()
        self.linear = nn.Linear(n_features,1)

    def forward(self, x):
        return self.linear(x)

In [ ]:
N_features = X_train_t.shape[1]

In [ ]:
model = LinearRegression(n_features=N_features)
model = model.to(device)
print(model)

In [ ]:
print(f"weight : {model.linear.weight.data}")
print(f"bias : {model.linear.bias.data}")

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(),lr = LEARNING_RATE)
N_epochs = 100

In [ ]:
train_losses = []
val_losses = []

for epoch in range(N_epochs):
    model.train()
    epoch_train_loss = 0
    
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        predictions = model(X_batch)
        l2_penalty = 0
        for p in model.parameters():
            l2_penalty += (p**2).sum()
        loss = criterion(predictions, y_batch) + l2_penalty * LAMBDA
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss/len(train_loader)
    train_losses.append(avg_train_loss)    

    model.eval()
    epoch_val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            val_prediction = model(X_batch)
            val_loss = criterion(val_prediction, y_batch)

            epoch_val_loss += val_loss.item()

        avg_val_loss = epoch_val_loss/len(val_loader)
        val_losses.append(avg_val_loss)

        print(f"for epoch {epoch}, training_loss: {avg_train_loss:.4f} and validation_loss: {avg_val_loss:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))

epochs_range = range(1, N_epochs + 1)

plt.plot(epochs_range, train_losses, label='Train MSE', color='steelblue',  linewidth=2)
plt.plot(epochs_range, val_losses,   label='Val MSE',   color='coral',      linewidth=2, linestyle='--')


plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Learning Curve — Linear Regression with Regularization (California Housing)')
plt.legend()
plt.grid(True, alpha=0.3)  
plt.tight_layout()

os.makedirs('../results/plots', exist_ok=True)
plt.savefig('../results/plots/linear_regression_learning_L2_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final Train MSE: {train_losses[-1]:.4f}')
print(f'Final Val   MSE: {val_losses[-1]:.4f}')

In [ ]:
model.eval()
all_preds = []
all_true  = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)

        all_preds.append(preds.cpu().numpy().flatten())
        all_true.append(y_batch.cpu().numpy().flatten())

y_pred_val = np.concatenate(all_preds)
y_true_val = np.concatenate(all_true)

mse  = np.mean((y_pred_val - y_true_val) ** 2)
rmse = np.sqrt(mse)
mae  = np.mean(np.abs(y_pred_val - y_true_val))

print(f'  MSE:  {mse:.4f} ')
print(f'  RMSE: {rmse:.4f}  ')
print(f'  MAE:  {mae:.4f} ')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))


axes[0].scatter(y_true_val, y_pred_val, alpha=0.3, s=8, color='steelblue')

lims = [min(y_true_val.min(), y_pred_val.min()),
        max(y_true_val.max(), y_pred_val.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
# 'r--' → red dashed line
axes[0].set_xlabel('True Value ($100k)')
axes[0].set_ylabel('Predicted Value ($100k)')
axes[0].set_title('Predicted vs True — Validation Set')
axes[0].legend()

residuals = y_pred_val - y_true_val
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linewidth=1.5, label='Zero residual')
axes[1].axvline(residuals.mean(), color='red', linestyle='--',
                label=f'Mean residual = {residuals.mean():.3f}')
axes[1].set_xlabel('Residual (Predicted − True)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')
axes[1].legend()

plt.suptitle('Linear Regression with L2 Regularization — Test Set Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/plots/linear_regression_Ridge_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs('../results/models', exist_ok=True)
os.makedirs('../results/logs',   exist_ok=True)

model_path = '../results/models/linear_regression_L2_regularization_california.pth'
torch.save(model.state_dict(), model_path)
print(f'Model saved to: {model_path}')

In [ ]:
log_path = '../results/logs/experiment_log.csv'

result_row = {
    'experiment':  'linear_regression',
    'dataset':     'california_housing',
    'model_type':  'Ridge',
    'degree_M':    1,              
    'lambda':      LAMBDA,           
    'n_epochs':    N_epochs,
    'batch_size':  BATCH_SIZE,
    'lr':          LEARNING_RATE,
    'train_mse':   round(train_losses[-1], 4),
    'val_mse':     round(val_losses[-1],   4),
    'test_rmse':    round(rmse,             4),
    'test_mae':     round(mae,              4),
    'test_mse':   round(mse,              4) ,          
}

result_df = pd.DataFrame([result_row])
if os.path.exists(log_path):
    result_df.to_csv(log_path, mode='a', header=False, index=False)
else:
    result_df.to_csv(log_path, mode='w', header=True, index=False)
